## 电影域交互数据预处理（2023版）
流程：加载 → 仅保留有meta的物品 → 迭代式双向质量过滤 → 保存中间结果

In [1]:
import json, gzip, csv, os
from collections import Counter
import pandas as pd

REVIEW_PATH = "./Movies_and_TV.jsonl.gz"
META_PATH = "./processed/movie_meta_clean.csv"
SAVE_DIR = "./processed/"
os.makedirs(SAVE_DIR, exist_ok=True)

In [2]:
# 加载干净meta（有title+imageURL的物品ID集合）
import sys
import csv
max_int = sys.maxsize
while True:
    try:
        csv.field_size_limit(max_int)
        break
    except OverflowError:
        max_int = max_int // 10
with open(META_PATH, encoding='utf-8') as f:
    meta_items = set()
    for row in csv.DictReader(f):
        meta_items.add(row['parent_asin'])
print(f'有meta的电影物品: {len(meta_items)}')

有meta的电影物品: 434061


In [3]:
# 遍历交互数据，仅做meta过滤
rows = []
total = 0
has_meta = 0

with gzip.open(REVIEW_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        total += 1
        try:
            obj = json.loads(line.strip())
        except:
            continue
        
        parent_asin = obj.get("parent_asin", "")
        if parent_asin not in meta_items:
            continue
        has_meta += 1
        
        rows.append({
            "user_id": obj["user_id"],
            "item_id": parent_asin,
            "timestamp": int(float(obj.get("timestamp", 0)) / 1000.0),
            "domain": "movie"
        })
        
        if total % 2000000 == 0:
            print(f"  已处理 {total/1e6:.1f}M, 有meta {has_meta} ...")

print(f"总交互: {total}")
print(f"有meta: {has_meta} ({has_meta/total*100:.1f}%)")

df = pd.DataFrame(rows)
del rows
print(f"当前: {len(df)} 条交互, {df['user_id'].nunique()} 用户, {df['item_id'].nunique()} 物品")

  已处理 2.0M, 有meta 983959 ...
  已处理 4.0M, 有meta 1950912 ...
  已处理 6.0M, 有meta 2898278 ...
  已处理 14.0M, 有meta 6354678 ...
  已处理 16.0M, 有meta 7239596 ...
总交互: 17328314
有meta: 7907205 (45.6%)
当前: 7907205 条交互, 3299815 用户, 433730 物品


In [4]:
# ===== 迭代式双向过滤 =====
MIN_ITEM = 5
MIN_USER = 5

while True:
    n_u_before = df["user_id"].nunique()
    n_i_before = df["item_id"].nunique()
    
    # 筛物品
    item_cnt = df.groupby("item_id").size()
    valid_items = item_cnt[item_cnt >= MIN_ITEM].index
    df = df[df["item_id"].isin(valid_items)]
    
    # 筛用户
    user_cnt = df.groupby("user_id").size()
    valid_users = user_cnt[user_cnt >= MIN_USER].index
    df = df[df["user_id"].isin(valid_users)]
    
    # 稳定则退出
    if df["user_id"].nunique() == n_u_before and df["item_id"].nunique() == n_i_before:
        break
    print(f"迭代中... 用户={df['user_id'].nunique()}, 物品={df['item_id'].nunique()}")

print(f"===== 电影域过滤完成 =====")
print(f"交互: {len(df)}")
print(f"用户: {df['user_id'].nunique()}")
print(f"物品: {df['item_id'].nunique()}")


迭代中... 用户=254100, 物品=160579
迭代中... 用户=239651, 物品=109041
迭代中... 用户=238544, 物品=106651
迭代中... 用户=238424, 物品=106422
迭代中... 用户=238406, 物品=106391
迭代中... 用户=238403, 物品=106387
迭代中... 用户=238403, 物品=106386
===== 电影域过滤完成 =====
交互: 3114700
用户: 238403
物品: 106386


In [5]:
# 保存电影域中间结果（等图书域处理完后一起合并划分）
df.to_csv(f"{SAVE_DIR}/movie_interactions_filtered.csv", index=False)
print(f"已保存: {SAVE_DIR}/movie_interactions_filtered.csv")

已保存: ./processed//movie_interactions_filtered.csv
